# Purpose

Do pretraining on 5 bases (ACTG + M (mC)).

In [3]:
import methylbert
import torch
import transformers
#importlib.reload(methylbert)
#importlib.reload(methylbert.model)
#%load_ext autoreload
#%autoreload 2

In [ ]:
#!cat ~/5BaseTestrun/bams.txt
!cat /home/bauerste/5BaseTestrun/bamOneDLBCL.txt

/data/gidb/shared/datasets/MethylBERT/B_cell/bam/SRR10165874_1_bismark_bt2_pe.bam
/data/gidb/shared/datasets/MethylBERT/B_cell/bam/SRR10165875_1_bismark_bt2_pe.bam
/data/gidb/shared/datasets/MethylBERT/B_cell/bam/SRR10165876_1_bismark_bt2_pe.bam
/data/gidb/shared/datasets/MethylBERT/B_cell/bam/SRR10165877_1_bismark_bt2_pe.bam
/data/gidb/shared/datasets/MethylBERT/B_cell/bam/RR10165884_1_bismark_bt2_pe.bam
/data/gidb/shared/datasets/MethylBERT/B_cell/bam/RR10165885_1_bismark_bt2_pe.bam


In [3]:
import methylbert.data.genome

methylbert.data.genome.pretrain_data_preprocess_5base(
    f_ref="/home/bauerste/5BaseTestrun/hg38/GCF_000001405.26_GRCh38_genomic.fa",
    sc_dataset="/home/bauerste/5BaseTestrun/bamOneDLBCL.txt",
    f_output="/home/bauerste/5BaseTestrun/pretrain_data/pretrain_5base.txt",
    min_seq_len=150,
    max_reads=1000000
)

Loading reference genome...
Loaded 23 chromosomes.
Keys: ['1', '2', '3', '4', '5']
Found 1 BAM file(s).


Processing BAM files:   0%|          | 0/1 [00:00<?, ?it/s]

  Processed 0 reads, kept 0...
  Processed 100,000 reads, kept 97,552...
  Processed 200,000 reads, kept 195,147...
  Processed 300,000 reads, kept 292,713...
  Processed 400,000 reads, kept 390,219...
  Processed 500,000 reads, kept 487,808...
  Processed 600,000 reads, kept 585,346...
  Processed 700,000 reads, kept 682,883...
  Processed 800,000 reads, kept 780,419...
  Processed 900,000 reads, kept 878,021...
  Reached max_reads limit (1000000), stopping.
Total sequences written: 975553
Output written to: /home/bauerste/5BaseTestrun/pretrain_data/pretrain_5base.txt


'/home/bauerste/5BaseTestrun/pretrain_data/pretrain_5base.txt'

In [ ]:
from Bio import SeqIO
records = SeqIO.parse("/home/bauerste/5BaseTestrun/hg38/GCF_000001405.26_GRCh38_genomic.fa", "fasta")
for i, record in enumerate(records):
    print(record.id)
    if i > 5:
        break

NC_000001.11
NT_187361.1
NT_187362.1
NT_187363.1
NT_187364.1
NT_187365.1
NT_187366.1


In [1]:
import pysam

with open("/home/bauerste/5BaseTestrun/bamOneDLBCL.txt", "r") as f:
    bam_path = f.readline().strip()

print("BAM path:", bam_path)

aln = pysam.AlignmentFile(bam_path, "rb")
print("BAM references:", aln.references[:5])

for read in aln.fetch(until_eof=True):
    if not read.is_unmapped:
        print("read.reference_name:", read.reference_name)
        print("read.query_alignment_length:", read.query_alignment_length)
        print("has XM tag:", read.has_tag("XM"))
        break

aln.close()

BAM path: /data/gidb/shared/datasets/MethylBERT/DLBCL/bam/SRR10099822_1_bismark_bt2.bam
BAM references: ('1', '10', '11', '12', '13')
read.reference_name: 17
read.query_alignment_length: 151
has XM tag: True


In [5]:
from methylbert.data.vocab import MethylVocab
from methylbert.data.dataset import MethylBertPretrainDataset

vocab = MethylVocab(k=3)
print(f"Vocab size: {len(vocab)}")  # should be 130

dataset = MethylBertPretrainDataset(
    f_path="/home/bauerste/5BaseTestrun/pretrain_data/pretrain_5base.txt",
    vocab=vocab,
    seq_len=150,
    n_cores=1
)

print(f"Dataset size: {len(dataset)}")
sample = dataset[0]
print(f"bert_input shape: {sample['bert_input'].shape}")
print(f"bert_input dtype: {sample['bert_input'].dtype}")
print(f"First 20 token ids: {sample['bert_input'][:20]}")

# sanity check: decode back to tokens
decoded = vocab.from_seq(sample['bert_input'][:20].tolist())
print(f"Decoded: {decoded}")

Building Vocab
Vocab size: 130
Open data : <_io.TextIOWrapper name='/home/bauerste/5BaseTestrun/pretrain_data/pretrain_5base.txt' mode='r' encoding='UTF-8'>
Total number of sequences :  975553
Lines are processed


ValueError: setting an array element with a sequence. The requested array has an inhomogeneous shape after 2 dimensions. The detected shape was (975553, 150) + inhomogeneous part.